# Text Preprocessing Pipeline

This notebook demonstrates essential text preprocessing techniques for NLP:
- Lowercasing
- Removing punctuation
- Tokenization
- Stopword removal
- Stemming
- Lemmatization

We use sklearn's 20 Newsgroups dataset as our corpus.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re
from collections import Counter

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.datasets import fetch_20newsgroups

sns.set_style('whitegrid')
print('All imports successful.')

## 1. Load the Dataset

We load a subset of the 20 Newsgroups dataset (4 categories) to keep things manageable.

In [ ]:
categories = ['sci.space', 'rec.sport.baseball', 'comp.graphics', 'talk.politics.guns']
newsgroups = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))

documents = newsgroups.data
target_names = newsgroups.target_names

print(f'Loaded {len(documents)} documents across {len(target_names)} categories.')
print(f'Categories: {target_names}')
print(f'\n--- Sample document ---\n{documents[0][:500]}')

## 2. Step-by-Step Preprocessing

Let's apply each preprocessing step individually on a sample document to see the effect.

In [ ]:
sample = documents[0][:500]
print('ORIGINAL TEXT:')
print(sample)
print('\n' + '='*60)

### 2.1 Lowercasing

In [ ]:
text_lower = sample.lower()
print('LOWERCASED:')
print(text_lower)

### 2.2 Remove Punctuation

In [ ]:
text_no_punct = text_lower.translate(str.maketrans('', '', string.punctuation))
print('PUNCTUATION REMOVED:')
print(text_no_punct)

### 2.3 Tokenization

In [ ]:
tokens = word_tokenize(text_no_punct)
print(f'TOKENIZED ({len(tokens)} tokens):')
print(tokens[:30])

### 2.4 Stopword Removal

In [ ]:
stop_words = set(stopwords.words('english'))
tokens_no_stop = [t for t in tokens if t not in stop_words and len(t) > 1]
print(f'STOPWORDS REMOVED ({len(tokens)} -> {len(tokens_no_stop)} tokens):')
print(tokens_no_stop[:30])

### 2.5 Stemming

In [ ]:
stemmer = PorterStemmer()
tokens_stemmed = [stemmer.stem(t) for t in tokens_no_stop]
print('STEMMED:')
for orig, stemmed in list(zip(tokens_no_stop, tokens_stemmed))[:15]:
    marker = ' <--' if orig != stemmed else ''
    print(f'  {orig:20s} -> {stemmed}{marker}')

### 2.6 Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()
tokens_lemmatized = [lemmatizer.lemmatize(t) for t in tokens_no_stop]
print('LEMMATIZED:')
for orig, lemma in list(zip(tokens_no_stop, tokens_lemmatized))[:15]:
    marker = ' <--' if orig != lemma else ''
    print(f'  {orig:20s} -> {lemma}{marker}')

## 3. Reusable Preprocessing Function

Let's combine all the steps into a single reusable function.

In [ ]:
def preprocess_text(text, use_lemmatization=True):
    """
    Full text preprocessing pipeline.
    
    Parameters
    ----------
    text : str
        Raw input text.
    use_lemmatization : bool
        If True, apply lemmatization. If False, apply stemming.
    
    Returns
    -------
    list[str]
        Cleaned tokens.
    """
    # Lowercase
    text = text.lower()
    # Remove numbers and punctuation
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and short tokens
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # Lemmatize or stem
    if use_lemmatization:
        lem = WordNetLemmatizer()
        tokens = [lem.lemmatize(t) for t in tokens]
    else:
        st = PorterStemmer()
        tokens = [st.stem(t) for t in tokens]
    return tokens

# Test on a sample
result = preprocess_text(documents[0][:300])
print('Preprocessed tokens:', result[:20])

## 4. Process the Full Corpus and Visualize Word Frequencies

In [ ]:
# Process all documents (this may take a minute)
all_tokens = []
for doc in documents[:500]:  # Use first 500 docs for speed
    all_tokens.extend(preprocess_text(doc))

word_counts = Counter(all_tokens)
print(f'Total tokens: {len(all_tokens)}')
print(f'Unique tokens: {len(word_counts)}')
print(f'\nTop 20 words: {word_counts.most_common(20)}')

In [ ]:
# Bar chart of top 30 words
top_n = 30
top_words = word_counts.most_common(top_n)
words, counts = zip(*top_words)

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, top_n))
ax.barh(range(top_n), counts[::-1], color=colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels(words[::-1])
ax.set_xlabel('Frequency')
ax.set_title(f'Top {top_n} Most Frequent Words in Corpus')
plt.tight_layout()
plt.show()

In [ ]:
# Word frequency display (wordcloud-style sizing with matplotlib)
fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Word Frequency Display (size ~ frequency)', fontsize=16)

top_50 = word_counts.most_common(50)
max_count = top_50[0][1]
min_count = top_50[-1][1]

np.random.seed(42)
for word, count in top_50:
    size = 10 + 30 * (count - min_count) / (max_count - min_count + 1)
    x, y = np.random.uniform(0.05, 0.95), np.random.uniform(0.05, 0.95)
    color = plt.cm.plasma(np.random.uniform(0.2, 0.9))
    ax.text(x, y, word, fontsize=size, color=color,
            ha='center', va='center', fontweight='bold', alpha=0.8)

plt.tight_layout()
plt.show()

## 5. Compare Preprocessing Effects per Category

In [ ]:
# Word frequencies per category
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for idx, cat in enumerate(target_names):
    ax = axes[idx // 2][idx % 2]
    cat_docs = [documents[i] for i in range(len(documents)) if newsgroups.target[i] == idx]
    cat_tokens = []
    for doc in cat_docs[:150]:
        cat_tokens.extend(preprocess_text(doc))
    cat_counts = Counter(cat_tokens).most_common(15)
    if cat_counts:
        w, c = zip(*cat_counts)
        ax.barh(range(len(w)), c[::-1], color=plt.cm.Set2(idx / 4))
        ax.set_yticks(range(len(w)))
        ax.set_yticklabels(w[::-1])
    ax.set_title(cat, fontsize=12)
    ax.set_xlabel('Frequency')

plt.suptitle('Top Words per Category', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

In this notebook we:
1. Loaded the 20 Newsgroups dataset from sklearn.
2. Applied each preprocessing step individually: lowercasing, punctuation removal, tokenization, stopword removal, stemming, and lemmatization.
3. Built a reusable `preprocess_text()` function that combines all steps.
4. Visualized word frequencies across the entire corpus and per category.

This preprocessing pipeline is the foundation for any downstream NLP task such as text classification, topic modeling, or information retrieval.